# Fetal Health Analysis

Fetal and maternal health during birth are one of the main concerns of United Nations. Global maternal and fetal health outcomes have drastically improved in the last decades, however stark regional and socioeconomic disparities persist. According to WHO and UNICEF, most maternal deaths during childbirth are preventable with timely access to skilled obstetric care. Additionally data from UN IGME highlights that the period during birth and the first month after carries the highest risk for the fetus and newborn. This project strives to find which health markers have the biggest significance for the outcome of labour.


## Main hypothesis

Fetal heart rate markers (low variability, prolonged and late decelerations) differ significantly between normal and abnormal groups in both datasets, and the strength of these differences is greater than for maternal factors such as age and parity. pH<7.15 is a risk factor.


In [2]:
%matplotlib inline

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import wfdb
from pathlib import Path

The first dataset is CTU-UHB Intrapartum Cardiotocography Database - a collection of 552 CTGs from the Czech Technical University (CTU) in Prague and the University Hospital in Brno (UHB). *Each recording is up to 90 minutes long, and includes a fetal heart rate time series, a uterine contraction signal, and maternal, delivery, and fetal clinical details.*

In [14]:
data_dir = Path("data/ctu-uhb-ctgdb-1.0.0")
names = sorted(p.stem for p in data_dir.glob("*.hea"))
rows = []
for name in names:
    h = wfdb.rdheader(str(data_dir / name))
    info = {"record": name}
    for line in h.comments:
        line = line.lstrip("#").strip()
        if not line or line.startswith(("-", "!")):
            continue
        key, value = line.rsplit(None, 1)
        try:
            info[key] = float(value)
        except ValueError:
            info[key] = value
    rows.append(info)
df_clinical = pd.DataFrame(rows)

In [15]:
df_clinical

,record,pH,BDecf,pCO2,BE,Apgar1,Apgar5,NICU days,Seizures,HIE,...,Induced,I.stage,NoProgress,CK/KP,II.stage,Deliv. type,dbID,Rec. type,Pos. II.st.,Sig2Birth
0,1001,7.14,8.14,7.7,-10.5,6.0,8.0,0.0,0.0,0.0,...,0.0,232.0,0.0,0.0,20.0,1.0,974204.0,1.0,14400.0,0.0
1,1002,7.00,7.92,12.0,-12.0,8.0,8.0,0.0,0.0,0.0,...,0.0,220.0,0.0,0.0,20.0,1.0,975666.0,1.0,14400.0,0.0
2,1003,7.20,3.03,8.3,-5.6,7.0,9.0,0.0,0.0,0.0,...,1.0,215.0,0.0,0.0,15.0,1.0,982315.0,1.0,14400.0,0.0
3,1004,7.30,5.19,5.5,-6.4,8.0,9.0,0.0,0.0,0.0,...,1.0,55.0,0.0,0.0,10.0,1.0,983277.0,12.0,14400.0,0.0
4,1005,7.30,4.52,5.7,-5.8,9.0,10.0,0.0,0.0,0.0,...,1.0,114.0,0.0,0.0,15.0,1.0,983398.0,1.0,14400.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,2042,7.33,-0.50,6.6,-0.8,9.0,10.0,0.0,0.0,0.0,...,0.0,255.0,0.0,1.0,15.0,2.0,1072108.0,1.0,15482.0,0.0
548,2043,7.08,10.92,7.9,-13.3,8.0,9.0,0.0,0.0,0.0,...,0.0,260.0,0.0,1.0,25.0,2.0,1078408.0,1.0,14400.0,0.0
549,2044,7.02,9.13,10.6,-12.3,8.0,8.0,0.0,0.0,0.0,...,1.0,335.0,1.0,0.0,-1.0,2.0,1079432.0,1.0,-1.0,0.0
550,2045,7.03,8.91,10.4,-12.2,7.0,9.0,0.0,0.0,0.0,...,0.0,189.0,0.0,0.0,-1.0,2.0,1083243.0,1.0,-1.0,0.0


In [16]:
print(df_clinical["Sex"].value_counts(dropna=False))
print(df_clinical["Presentation"].value_counts(dropna=False))

Sex
1.0    286
2.0    266
Name: count, dtype: int64
Presentation
1.0    499
3.0     32
2.0     18
NaN      3
Name: count, dtype: int64


In [6]:
df_clinical.to_csv("clinical_ctu_chb.csv")

In [7]:
df_clinical.columns

Index(['record', 'pH', 'BDecf', 'pCO2', 'BE', 'Apgar1', 'Apgar5', 'NICU days',
       'Seizures', 'HIE', 'Intubation', 'Main diag.', 'Other diag.',
       'Gest. weeks', 'Weight(g)', 'Sex', 'Age', 'Gravidity', 'Parity',
       'Diabetes', 'Hypertension', 'Preeclampsia', 'Liq. praecox', 'Pyrexia',
       'Meconium', 'Presentation', 'Induced', 'I.stage', 'NoProgress', 'CK/KP',
       'II.stage', 'Deliv. type', 'dbID', 'Rec. type', 'Pos. II.st.',
       'Sig2Birth'],
      dtype='str')

Since the values in the dataframe have been extracting form a raw text file, containing no information about their format, the first step would be to transform them in the correct type.

In [8]:
cat_cols = ["Presentation", "Deliv. type", "Sex", "Diabetes",
            "Hypertension", "Preeclampsia", "Induced", "Meconium", "Rec. type"]
for c in cat_cols:
    df_clinical[c] = df_clinical[c].astype("category")

Numerical data can be turned into meaningful content in respect to its corresponding code.

In [17]:
df_clinical["Sex"] = df_clinical["Sex"].map({1.0 :"female" , 2: "male"})
df_clinical["Presentation"] = df_clinical["Presentation"].map({1: "normal",2: "breech",3: "other"})
df_clinical["Deliv. type"] = df_clinical["Deliv. type"].map({1: "vaginal",2: "cs"})
df_clinical["Rec. type"] = df_clinical["Rec. type"].map({1: "USG",2: "FECG",12: "combined"})

In [28]:
df_clinical.rename(columns={"BE": "Base excess"}, inplace=True)

/var/folders/fl/dqlmf5t178bghccm7nwd121h0000gn/T/ipykernel_10813/2883172611.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clinical.rename(columns={"BE": "Base excess"}, inplace=True)


Getting to know the dataset, finding missing value, etc.

In [29]:
miss = df_clinical.isnull().sum().to_frame("n_missing")
miss["pct"] = (miss["n_missing"] / df_clinical.shape[0]) * 100
miss.sort_values("pct", ascending=False)

,n_missing,pct
BDecf,11,1.992754
pCO2,11,1.992754
Base excess,11,1.992754
Gravidity,4,0.724638
Presentation,3,0.543478
Rec. type,2,0.362319
Weight(g),1,0.181159
record,0,0.000000
I.stage,0,0.000000
Pyrexia,0,0.000000


There could be cases in which the missing values can be hidden through placeholders, such as values that are impossible for the particular marker

In [30]:
df_clinical.iloc[:, [x for x in range (7)]].describe()

,pH,BDecf,pCO2,Base excess,Apgar1,Apgar5
count,552.000000,541.000000,541.000000,541.000000,552.000000,552.000000
mean,7.230054,4.601904,7.069501,-6.384104,8.262681,9.068841
std,0.105039,3.553392,1.556686,3.827455,1.624959,1.085613
min,6.850000,-3.400000,0.700000,-26.800000,1.000000,4.000000
25%,7.170000,2.490000,5.900000,-8.000000,8.000000,9.000000
50%,7.250000,4.130000,7.000000,-5.600000,9.000000,9.000000
75%,7.300000,6.140000,7.900000,-4.000000,9.000000,10.000000
max,7.470000,26.110000,12.300000,-0.200000,10.000000,10.000000


By looking at the minimum and maximum values of these specific markers, we can come to the conclusion that there are no hidden missing values.

Now let's ask the question what missingness actually means for this dataset.

In [42]:
df_clinical["BDecf_missing"] = df_clinical["BDecf"].isna()
print([df_clinical.groupby("BDecf_missing")[["pH", "pCO2", "Gest. weeks", "Weight(g)"]].agg(["mean", "std", "median"])])

[                     pH                       pCO2                   \
                   mean       std median      mean       std median   
BDecf_missing                                                         
False          7.232292  0.100581   7.25  7.069501  1.556686    7.0   
True           7.120000  0.219180   7.02       NaN       NaN    NaN   

              Gest. weeks                     Weight(g)                     
                     mean       std median         mean        std  median  
BDecf_missing                                                               
False           39.992606  1.127083   40.0  3399.324074  455.89700  3410.0  
True            40.090909  1.445998   40.0  3454.545455  411.68832  3610.0  ]


From this table we can deduct that **pCO2** and **BDecf** are missing together.

In [43]:
cols = ["BDecf", "pCO2", "Base excess"]
df_clinical[cols].isna().value_counts()

BDecf  pCO2   Base excess
False  False  False          541
True   True   True            11
Name: count, dtype: int64

In [46]:
df_clinical["th_missing"] = df_clinical[["BDecf", "pCO2", "Base excess"]].isna().all(axis=1)
df_clinical.groupby("th_missing")[["pH", "Apgar1", "Apgar5"]].agg(["count", "mean", "std", "median"])

pH                            Apgar1                             \
           count      mean       std median  count      mean       std median   
th_missing                                                                      
False        541  7.232292  0.100581   7.25    541  8.264325  1.627997    9.0   
True          11  7.120000  0.219180   7.02     11  8.181818  1.537412    8.0   

           Apgar5                             
            count      mean       std median  
th_missing                                    
False         541  9.066543  1.088326    9.0  
True           11  9.181818  0.981650    9.0

Here little, but noticeable differences in the mean and median are present. Such differences can be due to the small subset that represents the records with missing values. Another interpretation requires biology and chemistry knowledge.

**BDecf** stands for base deficit of extracellular fluid and measures metabolic acid accumulation in blood, assessed in umbilical cord blood gases at birth; evaluation of metabolic acidosis

**pCO2** measures the partial pressure of $CO_2$ in fetal blood. It is typically evaluated via umbilical cord blood gas analysis at delivery; evaluation of acid-base balance and placental gas exchange

**Base excess** measures metabolic acid-base balance via umbilical artery right after birth, it defines the amount of strong acid (in $mmol/L$) needed to return a standart pH of 7.4 at a pCO2 of 40 mmHg and 37ºC; evaluation of oxygen deprivation and metabolic acidosis

As we can see, these three measures are all related to acidity, measured after birth. The records with missing values for them have a lower median pH, so BDecf analyses underrepresent severely academic cases. In a small set of only 11 records with a standart deviation almost twice as big as that of the other subset the means are unstable, since a few cases can drive the difference.

In [54]:
missing = df_clinical[df_clinical["th_missing"]]
present = df_clinical[~df_clinical["th_missing"]]
missing["critical_pH"] = (missing["pH"]<7.15)
missing["critical_pH"].value_counts()


critical_pH
True     7
False    4
Name: count, dtype: int64